# ThaiOCRBench — current model baseline

Runs the **current OCR pipeline** — `PP-OCRv5_mobile_det` + `th_PP-OCRv5_mobile_rec`, pretrained and
not fine-tuned yet — on the 4 reading tasks of
[ThaiOCRBench](https://huggingface.co/datasets/typhoon-ai/ThaiOCRBench) (IJCNLP-AACL 2025,
CC-BY-SA-4.0). It scores the output with the benchmark's **official metrics** (BMFL / ANLS,
re-implemented from its eval code) plus our **CER** and **exact match**.

| Task | n | What the model must do |
|---|---:|---|
| Fine-grained text recognition | 206 | read the text inside a given box, the closest task to a label field |
| Text recognition | 333 | read the text on a sign or photo |
| Full-page OCR | 197 | read a whole document page |
| Handwritten content extraction | 209 | handwriting (outside the BRD guarantee; informative) |

**Where to run it**

| Environment | How |
|---|---|
| Docker (on-prem) | `make notebook`, then open http://127.0.0.1:8888 with the token `NOTEBOOK_TOKEN` from `.env` |
| Google Colab | open this file from GitHub in Colab; cells 1–2 clone the repo and install dependencies |
| VS Code / local Jupyter | any Python 3.12 environment; cell 2 installs what is missing |

> This notebook only touches **public data and public pretrained models**. Never open real parcel
> images on Colab or any other external service: label images must stay inside the organization.

Speed: a GPU (e.g. Colab T4) is picked up automatically. On CPU expect about 1–3 s per image on a
modern x86 machine; start with `LIMIT = 20`.

In [ ]:
# 1 · Locate the repo (or clone it on Colab) and make `src` importable
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ntuankiet1/OCR.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo() -> Path | None:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "benchmarks" / "thaiocrbench.py").exists():
            return p
    return None


ROOT = find_repo()
if ROOT is None:
    target = Path("/content/OCR" if IN_COLAB else "OCR").resolve()
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
    ROOT = target
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT, "| colab:", IN_COLAB)

In [ ]:
# 2 · Dependencies: already in the Docker image, installed on the fly elsewhere (e.g. Colab)
# PaddlePaddle comes in two wheels: `paddlepaddle` (CPU only, never uses a GPU) and
# `paddlepaddle-gpu` (CUDA). A visible NVIDIA GPU (Colab T4, ...) gets the GPU wheel.
# Pinned to 3.2.2: 3.3.x fails on CPU with "ConvertPirAttribute2RuntimeAttribute not support"
# (oneDNN bug, Paddle #77340).
import importlib.util
import shutil

PADDLE_VERSION = "3.2.2"
PADDLE_GPU_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"
HAS_GPU = bool(shutil.which("nvidia-smi")) and subprocess.run(
    ["nvidia-smi"], capture_output=True
).returncode == 0
pip = [sys.executable, "-m", "pip", "install", "-q"]


def installed(dist: str) -> str | None:
    from importlib.metadata import PackageNotFoundError, version

    try:
        return version(dist)
    except PackageNotFoundError:
        return None


want = "paddlepaddle-gpu" if HAS_GPU else "paddlepaddle"
other = "paddlepaddle" if HAS_GPU else "paddlepaddle-gpu"
if installed(want) != PADDLE_VERSION or installed(other):
    already_loaded = "paddle" in sys.modules
    if installed(other):  # both wheels provide `import paddle`: never keep both
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", other], check=True)
    extra = ["-i", PADDLE_GPU_INDEX] if HAS_GPU else []
    print(f"installing {want}=={PADDLE_VERSION} (GPU visible: {HAS_GPU})")
    subprocess.run([*pip, f"{want}=={PADDLE_VERSION}", *extra], check=True)
    if already_loaded:
        raise RuntimeError("paddle changed in a running kernel: restart the runtime, then run all")

modules = ("paddleocr", "nltk", "pythainlp", "rapidfuzz")
missing = [m for m in modules if importlib.util.find_spec(m) is None]
if missing:
    print("installing for:", missing)
    subprocess.run([*pip, "paddleocr==3.7.0", "nltk", "pythainlp", "rapidfuzz"], check=True)

import nltk  # noqa: E402
import paddle  # noqa: E402

for resource, package in (("corpora/wordnet", "wordnet"), ("corpora/omw-1.4", "omw-1.4")):
    try:
        nltk.data.find(resource)  # METEOR needs WordNet (baked into the Docker image)
    except LookupError:
        nltk.download(package, quiet=True)
cuda = paddle.device.is_compiled_with_cuda()
print(f"paddle {paddle.__version__} · CUDA build: {cuda} · GPUs: "
      f"{paddle.device.cuda.device_count() if cuda else 0} · python {sys.version.split()[0]}")

## Configuration
`LIMIT` takes the first *N* samples per task, sorted by Id, so repeated runs see the same subset.
Set `LIMIT = None` for the full benchmark (945 images).

In [ ]:
# 3 · Configuration
import yaml

LIMIT = 20  # samples per task; None = full run
TASKS = [
    "Fine-grained text recognition",
    "Text recognition",
    "Full-page OCR",
    "Handwritten content extraction",
]

base = yaml.safe_load((ROOT / "params.yaml").read_text())["base_models"]
MODELS = {k: base[k] for k in ("det", "rec", "doc_orientation", "textline_orientation")}
# Use the pretrained files fetched by `make base-models` when present (the same files a bundle
# ships); otherwise PaddleOCR downloads the models by name.
BASE_DIR = ROOT / "artifacts" / "base"
MODELS_DIR = BASE_DIR if (BASE_DIR / "det").exists() else None
CACHE_DIR = ROOT / "data" / "benchmarks" / "thaiocrbench"
OUT_ROOT = ROOT / "reports" / "benchmarks" / "thaiocrbench"
print(MODELS)
print("models from:", MODELS_DIR or "PaddleOCR download by name")

## Data

In [ ]:
# 4 · Download the pinned dataset revision (cached) and load the 4 OCR tasks
import pandas as pd

from src.benchmarks import thaiocrbench as tb

shards = tb.download(CACHE_DIR)
df = tb.load(shards, TASKS, LIMIT)
print(f"{tb.REPO} @ {tb.REVISION[:8]} · {len(df)} samples")
df.groupby("Task").agg(
    n=("Id", "size"),
    categories=("category", "nunique"),
    median_answer_chars=("answer", lambda s: int(s.str.len().median())),
)

In [ ]:
# 5 · One sample per task: image (red box = region to read) + question + expected answer
import html

import cv2
import matplotlib.pyplot as plt
from IPython.display import HTML, display


def draw_box(img, question):
    box = tb.parse_bbox(question)
    if box is None:
        return img
    h, w = img.shape[:2]
    out = img.copy()
    p1 = (int(box[0] * w / 1000), int(box[1] * h / 1000))
    p2 = (int(box[2] * w / 1000), int(box[3] * h / 1000))
    cv2.rectangle(out, p1, p2, (0, 0, 255), max(2, w // 300))
    return out


def show_row(row, prediction=None, max_chars=400):
    """Image via matplotlib (no Thai text in the plot); text via HTML so Thai renders."""
    img = draw_box(tb.decode_image(row["image"]), row["question"])
    plt.figure(figsize=(7, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()
    cells = [("Task", row["Task"]), ("Category", row["category"]),
             ("Question", row["question"]), ("Answer", row["answer"])]
    if prediction is not None:
        cells.append(("Prediction", prediction))
    body = "".join(
        f"<tr><th style='text-align:left;vertical-align:top'>{k}</th>"
        f"<td style='text-align:left;white-space:pre-wrap'>"
        f"{html.escape(str(v)[:max_chars])}</td></tr>"
        for k, v in cells
    )
    display(HTML(f"<table>{body}</table>"))


for _, group in df.groupby("Task"):
    show_row(group.iloc[0])

## Model

In [ ]:
# 6 · Build the OCR engine (the first call loads or downloads the models), then warm up
import time

from src.inference.ocr import PaddleOcrEngine

label = f"base-{MODELS['rec']}" if MODELS_DIR else f"{MODELS['det']}+{MODELS['rec']}"
spec = tb.ModelSpec(models_dir=MODELS_DIR, label=label, **MODELS)

t0 = time.perf_counter()
engine = PaddleOcrEngine(
    spec.models_dir, spec.det, spec.rec, spec.doc_orientation, spec.textline_orientation
)
print(f"models loaded in {time.perf_counter() - t0:.1f}s on {engine.device}")

t0 = time.perf_counter()
lines = engine.read(tb.decode_image(df.iloc[0]["image"]))
print(f"warm-up: {len(lines)} text lines in {time.perf_counter() - t0:.2f}s")
print(tb.reading_order(lines)[:300])

## Run

In [ ]:
# 7 · Run the benchmark (prints progress every 25 samples; a failing sample scores 0, no crash)
t0 = time.perf_counter()
pred = tb.run(engine, df)
print(f"done: {len(pred)} samples in {(time.perf_counter() - t0) / 60:.1f} min")

## Results
**Official** is the benchmark's metric on a 0–1 scale, higher is better: ANLS for text recognition,
BMFL for the rest. **CER** is character error rate after our normalization (Thai digits, SARA AM,
tone-mark order), ignoring whitespace; lower is better.

In [ ]:
# 8 · Scores per task, then save the run (report.md, metrics.json, predictions.parquet)
summary = tb.summarize(pred)
report = tb.save_run(pred, summary, spec, LIMIT, OUT_ROOT)
print("saved:", report.parent)
columns = ["n", "official_metric", "official_score", "cer_mean", "cer_median",
           "exact_match", "latency_ms_p50", "latency_ms_p95", "errors"]
pd.DataFrame(summary["tasks"]).T[columns]

In [ ]:
# 9 · By document category (only cells with n ≥ 10, so small LIMIT runs show little here)
cat = pd.DataFrame(summary["by_category"])
if cat.empty:
    print("no category has ≥ 10 samples yet; run with a larger LIMIT")
else:
    display(cat.pivot(index="category", columns="task", values="official_score").round(3))

## Error analysis
Look for patterns: tone marks and upper/lower vowels, Thai vs Arabic digits, rotated or curved
text, small text, and reading order on full pages.

In [ ]:
# 10 · Worst samples per task by CER
WORST_N = 3
images = df.set_index("Id")["image"]
for task, p in pred.groupby("task"):
    display(HTML(f"<h3>{html.escape(task)}</h3>"))
    for r in p.sort_values("cer", ascending=False).head(WORST_N).itertuples():
        row = {"image": images[r.id], "question": r.question, "Task": r.task,
               "category": r.category, "answer": r.answer}
        note = f"[CER {r.cer:.2f} · official {r.official_score:.3f} · {r.mode}]"
        show_row(row, prediction=f"{r.prediction}\n\n{note}")

In [ ]:
# 11 · Most frequent character errors on the fine-grained task (short single-region answers)
from collections import Counter

from rapidfuzz.distance import Levenshtein

from src.text.normalize import normalize_common


def char(c: str) -> str:
    return "∅" if c == "∅" else f"{c!r} U+{ord(c):04X}"


errors = Counter()
for r in pred[pred["task"] == "Fine-grained text recognition"].itertuples():
    gt, pr = normalize_common(r.answer), normalize_common(r.prediction)
    for op in Levenshtein.editops(gt, pr):
        if op.tag == "replace":
            errors[(gt[op.src_pos], pr[op.dest_pos])] += 1
        elif op.tag == "delete":
            errors[(gt[op.src_pos], "∅")] += 1
        else:
            errors[("∅", pr[op.dest_pos])] += 1

pd.DataFrame(
    [{"expected": char(a), "predicted": char(b), "count": n}
     for (a, b), n in errors.most_common(20)]
)

## Compare runs
Re-run this notebook after fine-tuning (point `MODELS_DIR` at `artifacts/bundle/models`, or use
`make benchmark-thai ARGS="--bundle artifacts/bundle"`) to check that general Thai reading did not
degrade (forgetting). Only compare runs that used the same `LIMIT`.

In [ ]:
# 12 · All saved runs side by side (official score per task)
import json

runs = []
for f in sorted(OUT_ROOT.glob("*/metrics.json")):
    m = json.loads(f.read_text())
    for task, t in m["tasks"].items():
        runs.append({"run": f.parent.name, "limit": m.get("limit") or "full", "task": task,
                     "official": t["official_score"], "cer": t["cer_mean"]})
if runs:
    display(pd.DataFrame(runs).pivot_table(
        index=["run", "limit"], columns="task", values="official"
    ).round(3))
else:
    print("no saved runs yet")

## Reading the numbers
- **Same scale as the paper.** ThaiOCRBench's leaderboard covers vision-language models run
  zero-shot. Rough reference points from its Table 1 (column alignment in the extracted table may be
  off, so check the paper): Gemini 2.5 Pro ≈ 0.50 / 0.90 / 0.71 / 0.91 and Qwen2.5-VL-72B
  ≈ 0.15 / 0.72 / 0.39 / 0.75 for fine-grained / full-page / handwritten / text recognition.
- **BMFL penalizes short answers.** The BLEU part is 0 whenever the answer has fewer than 4 word
  tokens, so even a perfect short read tops out around 0.75. Judge the recognizer on the
  fine-grained task by **CER and exact match**.
- **Full-page scores also measure reading order.** Our row grouping is simple; multi-column pages
  lose points even when every word is right.
- **This is not parcel data.** Use it to choose between off-the-shelf models and to detect
  forgetting after fine-tuning. Acceptance (AC01) still needs our own 5,000 real labels.